In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

In [17]:
filename = 'NEWPCACORFILEFINAL.xlsx'

# Step A: Load the file as a single column (to bypass quote errors)
df_raw = pd.read_excel(filename, header=None)

In [18]:
df = df_raw[0].str.strip('"').str.split(',', expand=True)

In [19]:
# Step C: Assign correct column names and remove the header row
df.columns = ['SampleID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Tissue']
df = df.drop(0).reset_index(drop=True)

In [20]:
# Step D: Convert PC coordinates from text to numbers
for col in ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']:
    df[col] = pd.to_numeric(df[col])

# --- PREPARE TISSUE LABELS ---
# Removes '12yr' and '60yr' to leave only the tissue name
df['Target'] = df['Tissue'].str.replace('12yr', '', regex=False).str.replace('60yr', '', regex=False)

X = df[['PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
y = df['Target']

# --- MODEL TRAINING (LOOCV) ---
loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_pred.append(rf.predict(X.iloc[test_idx])[0])
    y_true.append(y.iloc[test_idx].values[0])

print(f"--- TISSUE MODEL RESULTS ---")
print(f"Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")

--- TISSUE MODEL RESULTS ---
Accuracy: 66.67%


In [21]:
df = df_raw[0].str.strip('"').str.split(',', expand=True)
df.columns = ['SampleID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Tissue']
df = df.drop(0).reset_index(drop=True)

for col in ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']:
    df[col] = pd.to_numeric(df[col])

# --- FILTER FOR SAMPLES WITH AGE ---
# Keep only rows that contain '12yr' or '60yr'
age_df = df[df['Tissue'].str.contains('12yr|60yr')].copy()
age_df['Target'] = age_df['Tissue'].apply(lambda x: '12yr' if '12yr' in x else '60yr')

X = age_df[['PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
y = age_df['Target']

# --- MODEL TRAINING (LOOCV) ---
loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_pred.append(rf.predict(X.iloc[test_idx])[0])
    y_true.append(y.iloc[test_idx].values[0])

print(f"--- AGE MODEL RESULTS ---")
print(f"Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")

--- AGE MODEL RESULTS ---
Accuracy: 75.00%


In [22]:
import pickle

# --- TO SAVE ---
# 'rf' is your trained RandomForestClassifier object
with open('tissue_model.pkl', 'wb') as file:
    pickle.dump(rf, file)

# --- TO LOAD ---
with open('tissue_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# Now you can use it immediately
# prediction = loaded_model.predict(new_data)

In [23]:
import pickle

# --- TO SAVE ---
# 'rf' is your trained RandomForestClassifier object
with open('age_model.pkl', 'wb') as file:
    pickle.dump(rf, file)

# --- TO LOAD ---
with open('age_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# Now you can use it immediately
# prediction = loaded_model.predict(new_data)